# FinanceBench: Evaluation Playground


<hr style="border-bottom:0.1px solid gray">

##### (1) API Requirements
Add the following API keys into your `.env` file:

```ruby
OPENAI_API_KEY = 'INSERT API KEY HERE'
ANTHROPIC_API_KEY = 'INSERT API KEY HERE'
REPLICATE_API_TOKEN = 'INSERT API KEY HERE'
```

##### (2) Required Folder Structure

```bash
|-- /
|    |-- data/
|    |      | -- financebench_open_source.jsonl
     |      | -- financebench_document_information.jsonl
|    |-- pdfs/
|           | -- <... provided filings as PDF documents ...>
|    |-- results/
|    |-- vectorstores/
|    |-- evaluation_playground.ipynb
```


<br>
<hr style="border-bottom:0.1px solid gray">

In [13]:
%%capture
!pip install -U pandas numpy tqdm python-dotenv pymupdf chromadb tiktoken openai langchain langchain-community langchain-openai langchain-text-splitters

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "YOUR-API-KEY"
os.environ["OPENAI_BASE_URL"] = "YOUR-BASE-URL"

In [8]:
import os
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv()

# LangChain 与 OpenAI 依赖
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

import openai
import tiktoken

# 可在启动前导出、写入 .env，或在前一单元设置。
# export OPENAI_API_KEY="sk-..."
# export OPENAI_BASE_URL="https://your-proxy.example.com/v1"
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
OPENAI_BASE_URL = os.environ.get("OPENAI_BASE_URL") or os.environ.get("OPENAI_API_BASE")

if not OPENAI_API_KEY:
    raise ValueError("Please set OPENAI_API_KEY in your environment or .env file.")

# 同时保留两个变量名，兼容中转服务。
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
if OPENAI_BASE_URL:
    os.environ["OPENAI_BASE_URL"] = OPENAI_BASE_URL
    os.environ["OPENAI_API_BASE"] = OPENAI_BASE_URL

openai.api_key = OPENAI_API_KEY
if OPENAI_BASE_URL:
    openai.base_url = OPENAI_BASE_URL

In [9]:
##############################################################################
# OpenAI 模型配置
##############################################################################
OPENAI_MODEL_NAME = "gpt-4o-mini"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

model_config = {
    "provider": "openai",
    "model_name": OPENAI_MODEL_NAME,
    "eval_mode": "singleStore",
    "temp": 0.01,
    "max_tokens": 2048,
}

##############################################################################
# 数据集配置
##############################################################################
PATH_CURRENT = os.path.abspath(os.getcwd())
PATH_DATASET_JSONL = PATH_CURRENT + "/data/financebench_open_source.jsonl"
PATH_DOCUMENT_INFO_JSONL = PATH_CURRENT + "/data/financebench_document_information.jsonl"
PATH_RESULTS = PATH_CURRENT + "/results/"
PATH_PDFS = PATH_CURRENT + "/pdfs/"

# 选择数据子集：
# - ALL: 全量数据
# - OPEN_SOURCE: 开源子集（n=150）
# - CLOSED_SOURCE: 闭源子集，需申请访问
DATASET_PORTION = "OPEN_SOURCE"

# 示例可设为整数；设为 None 则评测全部。
MAX_EVAL_QUESTIONS = 30

##############################################################################
# 向量库配置
##############################################################################
VS_CHUNK_SIZE = 1024
VS_CHUNK_OVERLAP = 30
VS_DIR_VS = PATH_CURRENT + f"/vectorstores/{OPENAI_EMBEDDING_MODEL}"

In [10]:
##############################################################################
# 加载数据集
##############################################################################

# 加载完整数据
df_questions = pd.read_json(PATH_DATASET_JSONL, lines=True)
df_meta = pd.read_json(PATH_DOCUMENT_INFO_JSONL, lines=True)
df_full = pd.merge(df_questions, df_meta, on="doc_name")

# 获取全部文档
df_questions = df_questions.sort_values('doc_name')
ALL_DOCS = df_questions['doc_name'].unique().tolist()
print(f"Total number of distinct PDF: {len(ALL_DOCS)}")

# 筛选数据子集
if DATASET_PORTION != "ALL":
    df_questions = df_questions.loc[df_questions["dataset_subset_label"]==DATASET_PORTION]

# 限制示例数量；设为 None 跑全量。
df_questions = df_questions.sort_values('doc_name').reset_index(drop=True)
if MAX_EVAL_QUESTIONS is not None:
    df_questions = df_questions.head(MAX_EVAL_QUESTIONS).copy()
print(f"Number of questions: {len(df_questions)}")

# 检查相关文档
docs = df_questions['doc_name'].unique().tolist()
print(f"Number of distinct PDF in evaluation subset: {len(docs)}")

Total number of distinct PDF: 84
Number of questions: 30
Number of distinct PDF in evaluation subset: 16


In [11]:
##############################################################################
# 辅助函数：PDF 解析与向量库
##############################################################################
def get_pdf_text(doc):

    path_doc = f"{PATH_PDFS}/{doc}.pdf"
    pdf_reader = PyMuPDFLoader(path_doc)
    pdf_text = pdf_reader.load()

    return pdf_text

def get_openai_embeddings():
    return OpenAIEmbeddings(
        model=OPENAI_EMBEDDING_MODEL,
        api_key=OPENAI_API_KEY,
        base_url=OPENAI_BASE_URL,
    )

def build_vectorstore_retriever(docs, embeddings=None, top_k=4):

    if embeddings is None:
        embeddings = get_openai_embeddings()

    if docs == "all":
        docs = ALL_DOCS
        db_path = VS_DIR_VS + "/shared"
    else:
        docs = [docs]
        db_path = VS_DIR_VS + "/" + docs[0]

    # 创建或加载向量库；缺少 Chroma DB 时重建。
    os.makedirs(db_path, exist_ok=True)
    if not os.path.exists(f"{db_path}/chroma.sqlite3"):
        vectordb = Chroma(persist_directory=db_path, embedding_function=embeddings)
        vectordb.persist()

        # 写入文档到向量库
        for doc in docs:
            pdf_text = get_pdf_text(doc)
            text_splitter = RecursiveCharacterTextSplitter(
                chunk_size = VS_CHUNK_SIZE,
                chunk_overlap = VS_CHUNK_OVERLAP,
            )
            splitted_texts = text_splitter.split_documents(pdf_text)

            # 写入向量库
            vectordb.add_documents(documents=splitted_texts)
            vectordb.persist()

    else:
        vectordb = Chroma(persist_directory=db_path, embedding_function=embeddings)

    return vectordb.as_retriever(search_kwargs={"k": top_k}), vectordb

##############################################################################
# 模型与调用函数
##############################################################################

def get_max_context_length(prompt, openai_cutoff=105000):

    try:
        tokenizer_openai = tiktoken.encoding_for_model(OPENAI_MODEL_NAME)
    except KeyError:
        tokenizer_openai = tiktoken.get_encoding("cl100k_base")

    tokens_openai = tokenizer_openai.encode(prompt)
    number_of_chars_openai = len(prompt)

    if len(tokens_openai) > openai_cutoff:
        tokens_openai_tokens = [tokenizer_openai.decode_single_token_bytes(token) for token in tokens_openai]
        token_lengths_openai = [len(token) for token in tokens_openai_tokens]
        number_of_chars_openai = sum(token_lengths_openai[:openai_cutoff])

    return number_of_chars_openai

def get_model(provider="openai", model_name=OPENAI_MODEL_NAME, temp=0.01, max_tokens=2048):

    if provider != "openai":
        raise ValueError("This notebook is configured for OpenAI-only evaluation.")

    return ChatOpenAI(
        model=model_name,
        temperature=temp,
        max_tokens=max_tokens,
        api_key=OPENAI_API_KEY,
        base_url=OPENAI_BASE_URL,
    )


def _model_response_text(response):
    return getattr(response, "content", response)


def get_answer(model, eval_mode, question, context, retriever, retriever_only=False):

    retrieved_documents = []

    if eval_mode == "closedBook":
        prompt = f"Answer this question: {question}"
        answer = _model_response_text(model.invoke(prompt))

    elif eval_mode == "oracle":
        prompt = f"Answer this question: {question} \nHere is the relevant evidence that you need to answer the question:\n[START OF FILING] {context} [END OF FILING]"
        answer = _model_response_text(model.invoke(prompt))

    elif eval_mode == "oracle_reverse":
        prompt = f"Context:\n[START OF FILING] {context} [END OF FILING]\n\n Answer this question: {question} \n"
        answer = _model_response_text(model.invoke(prompt))

    elif eval_mode in ["inContext",  "inContext_reverse"]:
        # 截断上下文以满足 token 限制
        max_number_of_chars = get_max_context_length(context)
        context = context[:max_number_of_chars]

        if eval_mode == "inContext":
            prompt = f"Answer this question: {question} \nHere is the relevant filing that you need to answer the question:\n[START OF FILING] {context} [END OF FILING]"
        else:
            prompt = f"Context:\n[START OF FILING] {context} [END OF FILING]\n\n Answer this question: {question}\n"

        answer = _model_response_text(model.invoke(prompt))

    elif eval_mode == "singleStore" or eval_mode == "sharedStore":
        prompt = f"{question}"
        retrieved_documents = retriever.invoke(prompt)

        if retriever_only or not model:
            return ("", retrieved_documents)

        context = "\n\n".join(doc.page_content for doc in retrieved_documents)
        qa_prompt = (
            f"Answer this question using only the retrieved context.\n"
            f"Question: {question}\n\n"
            f"[START OF RETRIEVED CONTEXT]\n{context}\n[END OF RETRIEVED CONTEXT]"
        )
        answer = _model_response_text(model.invoke(qa_prompt))

    else:
        raise ValueError("Unknown 'eval_mode'!")

    return (answer, retrieved_documents)


In [12]:
##############################################################################
# 检索评测（不输出 CSV）
##############################################################################

# 选择检索模式：
# - singleStore: 每个文件单独建库和查询
# - sharedStore: 所有文件共用一个向量库
RETRIEVAL_EVAL_MODE = model_config["eval_mode"]
RETRIEVAL_KS = [5, 10]
MAX_RETRIEVAL_K = max(RETRIEVAL_KS)

# 设置评测问题
df_eval = df_questions


def get_relevant_evidence_pages(row):
    """Return zero-indexed evidence pages as (doc_name, page_num) pairs."""
    relevant_pages = set()
    for evidence in row["evidence"]:
        evidence_doc_name = (
            evidence.get("evidence_doc_name")
            or evidence.get("doc_name")
            or row["doc_name"]
        )
        evidence_page_num = evidence.get("evidence_page_num")
        if evidence_doc_name is not None and evidence_page_num is not None:
            relevant_pages.add((evidence_doc_name, int(evidence_page_num)))
    return relevant_pages


def get_retrieved_page(document):
    """Map a retrieved LangChain Document back to (doc_name, page_num)."""
    metadata = document.metadata or {}
    source = metadata.get("source", "")
    doc_name = metadata.get("doc_name") or os.path.splitext(os.path.basename(source))[0]
    page_num = metadata.get("page", metadata.get("page_number"))

    if not doc_name or page_num is None:
        return None

    return (doc_name, int(page_num))


def score_retrieval(retrieved_documents, relevant_pages, ks):
    retrieved_pages = [get_retrieved_page(doc) for doc in retrieved_documents]
    retrieved_pages = [page for page in retrieved_pages if page is not None]

    metrics = {}
    for k in ks:
        top_k_pages = set(retrieved_pages[:k])
        metrics[f"recall@{k}"] = len(top_k_pages & relevant_pages) / len(relevant_pages)

    first_relevant_rank = next(
        (rank for rank, page in enumerate(retrieved_pages, start=1) if page in relevant_pages),
        None,
    )
    metrics["MRR"] = 0.0 if first_relevant_rank is None else 1.0 / first_relevant_rank
    metrics["first_relevant_rank"] = first_relevant_rank
    return metrics, retrieved_pages


if RETRIEVAL_EVAL_MODE not in ["singleStore", "sharedStore"]:
    raise ValueError("RETRIEVAL_EVAL_MODE must be either 'singleStore' or 'sharedStore'.")

retriever_cache = {}
retrieval_results = []

for _, row in tqdm(df_eval.sort_values("doc_name").iterrows(), total=len(df_eval)):
    docs = row["doc_name"] if RETRIEVAL_EVAL_MODE == "singleStore" else "all"

    if docs not in retriever_cache:
        retriever_cache[docs], _ = build_vectorstore_retriever(
            docs=docs,
            top_k=MAX_RETRIEVAL_K,
        )

    retrieved_documents = retriever_cache[docs].invoke(row["question"])
    relevant_pages = get_relevant_evidence_pages(row)
    metrics, retrieved_pages = score_retrieval(retrieved_documents, relevant_pages, RETRIEVAL_KS)

    retrieval_results.append({
        "financebench_id": row["financebench_id"],
        "doc_name": row["doc_name"],
        "question": row["question"],
        "num_relevant_pages": len(relevant_pages),
        "relevant_pages": sorted(relevant_pages),
        "retrieved_pages": retrieved_pages[:MAX_RETRIEVAL_K],
        **metrics,
    })


df_retrieval_eval = pd.DataFrame(retrieval_results)
metric_cols = [f"recall@{k}" for k in RETRIEVAL_KS] + ["MRR"]
summary = df_retrieval_eval[metric_cols].mean()

print(f"OpenAI chat model configured: {OPENAI_MODEL_NAME}")
print(f"OpenAI embedding model: {OPENAI_EMBEDDING_MODEL}")
print(f"OpenAI base URL: {OPENAI_BASE_URL or 'default OpenAI endpoint'}")
print(f"Retrieval evaluation mode: {RETRIEVAL_EVAL_MODE}")
print(f"Number of questions: {len(df_retrieval_eval)}")
print(summary.to_string(float_format=lambda x: f"{x:.4f}"))

# 在 notebook 中展示逐题明细，不写 CSV。
df_retrieval_eval

100%|██████████| 30/30 [00:27<00:00,  1.07it/s]

OpenAI chat model configured: gpt-4o-mini
OpenAI embedding model: text-embedding-3-small
OpenAI base URL: https://api.chatanywhere.tech/v1
Retrieval evaluation mode: singleStore
Number of questions: 30
recall@5    0.3333
recall@10   0.5000
MRR         0.2460


,financebench_id,doc_name,question,num_relevant_pages,relevant_pages,retrieved_pages,recall@5,recall@10,MRR,first_relevant_rank
0,financebench_id_03029,3M_2018_10K,What is the FY2018 capital expenditure amount ...,1,"[(3M_2018_10K, 59)]","[(3M_2018_10K, 38), (3M_2018_10K, 46), (3M_201...",0.0,0.0,0.000000,NaN
1,financebench_id_04672,3M_2018_10K,Assume that you are a public equities analyst....,1,"[(3M_2018_10K, 57)]","[(3M_2018_10K, 126), (3M_2018_10K, 40), (3M_20...",0.0,0.0,0.000000,NaN
2,financebench_id_00499,3M_2022_10K,Is 3M a capital-intensive business based on FY...,3,"[(3M_2022_10K, 47), (3M_2022_10K, 49), (3M_202...","[(3M_2022_10K, 18), (3M_2022_10K, 33), (3M_202...",0.0,0.0,0.000000,NaN
3,financebench_id_01226,3M_2022_10K,What drove operating margin change as of FY202...,1,"[(3M_2022_10K, 26)]","[(3M_2022_10K, 19), (3M_2022_10K, 18), (3M_202...",0.0,0.0,0.000000,NaN
4,financebench_id_01865,3M_2022_10K,"If we exclude the impact of M&A, which segment...",1,"[(3M_2022_10K, 24)]","[(3M_2022_10K, 31), (3M_2022_10K, 31), (3M_202...",0.0,0.0,0.000000,NaN
5,financebench_id_00807,3M_2023Q2_10Q,Does 3M have a reasonably healthy liquidity pr...,1,"[(3M_2023Q2_10Q, 4)]","[(3M_2023Q2_10Q, 70), (3M_2023Q2_10Q, 69), (3M...",0.0,0.0,0.000000,NaN
6,financebench_id_00941,3M_2023Q2_10Q,Which debt securities are registered to trade ...,1,"[(3M_2023Q2_10Q, 0)]","[(3M_2023Q2_10Q, 69), (3M_2023Q2_10Q, 69), (3M...",0.0,1.0,0.166667,6.0
7,financebench_id_01858,3M_2023Q2_10Q,Does 3M maintain a stable trend of dividend di...,1,"[(3M_2023Q2_10Q, 61)]","[(3M_2023Q2_10Q, 72), (3M_2023Q2_10Q, 69), (3M...",1.0,1.0,0.333333,3.0
8,financebench_id_02987,ACTIVISIONBLIZZARD_2019_10K,What is the FY2019 fixed asset turnover ratio ...,2,"[(ACTIVISIONBLIZZARD_2019_10K, 68), (ACTIVISIO...","[(ACTIVISIONBLIZZARD_2019_10K, 48), (ACTIVISIO...",0.0,0.5,0.111111,9.0
9,financebench_id_07966,ACTIVISIONBLIZZARD_2019_10K,What is the FY2017 - FY2019 3 year average of ...,2,"[(ACTIVISIONBLIZZARD_2019_10K, 69), (ACTIVISIO...","[(ACTIVISIONBLIZZARD_2019_10K, 41), (ACTIVISIO...",0.0,0.0,0.000000,NaN
